In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_RBF.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 2.0, 'n_it': 2.0}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 300

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[15.124024495789012, 14.782028472385637, 14.948904262850956, 15.115723168545884, 14.566188168291076, 15.127755164003743, 14.861248628355586, 15.080150750165211, 14.643638592953593, 13.996335030068975, 14.52648154191649, 14.56421501879236, 14.96904829748126, 14.321767172815047, 14.78570789018349, 13.956800240479204, 14.250061198940053, 14.9808872802236, 15.082697451864519, 14.765115088112147, 14.763464154293965, 14.385326629144897, 14.672044087183096, 14.226469474225345, 14.884597841476456, 15.053560289600068, 15.059334589223758, 15.072838721233243, 15.133273409755054, 14.847669852312674, 14.882038957382564, 14.554615427426311, 15.07538353402289, 14.293191012869555, 14.82523989508276, 14.351353257850269, 14.8614790624905, 15.030890977357677, 14.63332156508589, 14.974339390546483, 14.938733565328768, 14.72083829277553, 15.018980999872493, 14.649642392840928, 14.868527086032941, 14.739140703379892, 14.917852006906365, 14.802173826248392, 15.14258201664029, 14.436288855688769, 14.826741697

In [5]:
np.average(y_max_arr)

np.float64(14.768516833381987)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_RBF/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)